<a href="https://colab.research.google.com/github/felpscunha/projects_datascience/blob/main/portoseguro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bibliotecas e Arquivos

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# scikit-learn libraries

from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# XGBOOST

from xgboost import XGBClassifier
import xgboost as xgb

In [ ]:
df_train = pd.read_csv('/content/train.csv')
df_test = pd.read_csv('/content/test.csv')
metadata = pd.read_csv('/content/metadata.csv')
submisssion = pd.read_csv('/content/submission_sample.csv')

# An?lise Explorat?ria dos Dados

In [ ]:
df_train.head(1)

In [ ]:
df_train.shape

In [ ]:
df_train.info()

In [ ]:
df_train.duplicated().sum()

# Outliers

In [ ]:
# outliers detection with z-score
from scipy import stats

z_score = np.abs(stats.zscore(df_train))
print(z_score)

In [ ]:
outliers_z = np.where(z_score > 3)

In [ ]:
df_train_out_z = df_train[(z_score < 3).all(axis=1)]

In [ ]:
df_train_out_z.shape

# Balanceamento dos Dados

In [ ]:
sns.countplot(df_train['y'])

In [ ]:
from imblearn.over_sampling import RandomOverSampler

oversampler = RandomOverSampler(sampling_strategy='minority', random_state=5)
x_over, y_over = oversampler.fit_resample(df_train.drop(['id','y'], axis=1), df_train['y'])

In [ ]:
df_train_over = pd.DataFrame(x_over)
df_train_over.shape

In [ ]:
sns.countplot(y_over)

## Balanceamento Parte 2

In [ ]:
from imblearn.over_sampling import BorderlineSMOTE

bsmoter = BorderlineSMOTE(sampling_strategy='minority', random_state=5)
x_over, y_over = bsmoter.fit_resample(df_train_out_z.drop(['id','y'], axis=1), df_train_out_z['y'])

In [ ]:
df_train_over = pd.DataFrame(x_over)
df_train_over.shape

In [ ]:
sns.countplot(y_over)

# Modelo + Pr?-processamentos

In [ ]:
X = df_train_over.drop(['id', 'y'], axis = 1)
y = df_train_over['y']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_train_over, y_over, test_size=.20, random_state=5)

In [ ]:
rfc = RandomForestClassifier(random_state=5)
rfc.fit(X_train, y_train)
predictions_1st = rfc.predict(X_test)

In [ ]:
print(f'Baseline model accuracy: {accuracy_score(y_test, predictions_1st)}')
print(f'Baseline model f1-score: {f1_score(y_test, predictions_1st)}')

# Scaling variables

In [ ]:
feat_importances = pd.Series(rfc.feature_importances_, index=X_train.columns)
feat_importances.nlargest(10).plot(kind='barh')
plt.show()

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [ ]:
X_train_scaled = pd.DataFrame(X_train_scaled)
X_test_scaled = pd.DataFrame(X_test_scaled)

In [ ]:
X_test_scaled.head(1)

In [ ]:
rfc = RandomForestClassifier(random_state=5)
rfc.fit(X_train_scaled, y_train)
predictions_2nd = rfc.predict(X_test_scaled)

In [ ]:
print(f'Scaled model accuracy: {accuracy_score(y_test, predictions_2nd)}')
print(f'Scaled model f1-score: {f1_score(y_test, predictions_2nd)}')

# XGBOOST

In [ ]:
xgb_model = XGBClassifier(random_state = 42)
xgb_model.fit(X_train, y_train)

In [ ]:
predictions_3nd = xgb_model.predict(X_test)

In [ ]:
print(f'XGB model accuracy: {accuracy_score(y_test, predictions_3nd)}')
print(f'XGB model f1-score: {f1_score(y_test, predictions_3nd)}')

In [ ]:
xgb_pred = xgb_model.predict(df_test_copy)

In [ ]:
ids = df_test['id']
submission = {'id':ids, 'predicted':xgb_pred}
xgb_submission = pd.DataFrame(submission)

In [ ]:
xgb_submission.to_csv('submission_05.csv', index=False)

# XGBOOST Scaling variables

In [ ]:
xgb_model.fit(X_train_scaled, y_train)

In [ ]:
predictions_4nd = xgb_model.predict(X_test_scaled)

In [ ]:
print(f'XGB model w/ scaling accuracy: {accuracy_score(y_test, predictions_4nd)}')
print(f'XGB model w/ scaling f1-score: {f1_score(y_test, predictions_4nd)}')

In [ ]:
xgb_pred_scaled = xgb_model.predict(df_test_scaled)

In [ ]:
ids = df_test['id']
submission = {'id':ids, 'predicted':xgb_pred_scaled}
xgb_scaled_submission = pd.DataFrame(submission)

In [ ]:
xgb_scaled_submission.to_csv('submission_09.csv', index=False)

# Boosting model

In [ ]:
a_grid = {'learning_rate':[0.1, 0.2, 0.3],
              'n_estimators':[100, 250, 500, 1000, 2000],
              'max_depth':[3, 5, 10]
          }

In [ ]:
a_grid

In [ ]:
grid_1 = GridSearchCV(estimator=xgb_model, param_grid=a_grid, scoring='roc_auc', n_jobs=-1, cv=3)
grid_1.fit(X_train_scaled, y_train)

In [ ]:
grid_1.best_params_, grid_1.best_score_

In [ ]:
b_grid = {'learning_rate':[0.05, 0.1],
              'n_estimators':[50, 100, 150],
              'max_depth':[2, 3, 4]
          }

In [ ]:
grid_2 = GridSearchCV(estimator=xgb_model, param_grid=b_grid, scoring='roc_auc', n_jobs=-1, cv=3)
grid_2.fit(X_train_scaled, y_train)

In [ ]:
grid_2.best_params_, grid_2.best_score_

In [ ]:
c_grid = {'n_estimators':[250, 300, 350], 'max_depth':[1, 2, 3]
          }

In [ ]:
grid_3 = GridSearchCV(estimator=xgb_model, param_grid=c_grid, scoring='roc_auc', n_jobs=-1, cv=3)
grid_3.fit(X_train_scaled, y_train)

In [ ]:
grid_3.best_params_, grid_3.best_score_

In [ ]:
d_grid = {'min_child_weight':[1, 3, 5], 'max_depth':[2, 3, 4]}

In [ ]:
grid_4 = GridSearchCV(estimator=XGBClassifier(base_score=0.5, booster='gbtree',
                                     colsample_bylevel=1, colsample_bynode=1,
                                     colsample_bytree=1, gamma=0,
                                     learning_rate=0.1, max_delta_step=0,
                                     max_depth=3, min_child_weight=1,
                                     missing=None, n_estimators=100, n_jobs=1,
                                     nthread=None, objective='binary:logistic',
                                     random_state=42, reg_alpha=0, reg_lambda=1,
                                     scale_pos_weight=1, seed=None, silent=None,
                                     subsample=1, verbosity=1), param_grid=d_grid, scoring='roc_auc', n_jobs=-1, cv=3)
grid_4.fit(X_train_scaled, y_train)

In [ ]:
grid_4.best_params_, grid_4.best_score_

In [ ]:
e_grid = {'gamma':[0.0, 0.1, 0.2, 0.3, 0.4]}

In [ ]:
grid_5 = GridSearchCV(estimator=XGBClassifier(base_score=0.5, booster='gbtree',
                                     colsample_bylevel=1, colsample_bynode=1,
                                     colsample_bytree=1, gamma=0,
                                     learning_rate=0.1, max_delta_step=0,
                                     max_depth=3, min_child_weight=3,
                                     missing=None, n_estimators=100, n_jobs=1,
                                     nthread=None, objective='binary:logistic',
                                     random_state=42, reg_alpha=0, reg_lambda=1,
                                     scale_pos_weight=1, seed=None, silent=None,
                                     subsample=1, verbosity=1), param_grid=e_grid, scoring='roc_auc', n_jobs=-1, cv=3)
grid_5.fit(X_train_scaled, y_train)

In [ ]:
grid_5.best_params_, grid_5.best_score_

In [ ]:
f_grid = {'subsample':[0.6, 0.7, 0.8, 0.9], 'colsample_bytree':[0.6, 0.7, 0.8, 0.9]} 

In [ ]:
grid_6 = GridSearchCV(estimator=XGBClassifier(base_score=0.5, booster='gbtree',
                                     colsample_bylevel=1, colsample_bynode=1,
                                     colsample_bytree=1, gamma=0.0,
                                     learning_rate=0.1, max_delta_step=0,
                                     max_depth=3, min_child_weight=3,
                                     missing=None, n_estimators=100, n_jobs=1,
                                     nthread=None, objective='binary:logistic',
                                     random_state=42, reg_alpha=0, reg_lambda=1,
                                     scale_pos_weight=1, seed=None, silent=None,
                                     subsample=1, verbosity=1), param_grid=f_grid, scoring='roc_auc', n_jobs=-1, cv=3)
grid_6.fit(X_train_scaled, y_train)

In [ ]:
grid_6.best_params_, grid_6.best_score_

In [ ]:
g_grid = {'reg_alpha':[0.0001, 0.001, 0.01, 0.1, 1, 10]} 

In [ ]:
grid_7 = GridSearchCV(estimator=XGBClassifier(base_score=0.5, booster='gbtree',
                                     colsample_bylevel=1, colsample_bynode=1,
                                     colsample_bytree=0.8, gamma=0.3,
                                     learning_rate=0.1, max_delta_step=0,
                                     max_depth=2, min_child_weight=3,
                                     missing=None, n_estimators=350, n_jobs=1,
                                     nthread=None, objective='binary:logistic',
                                     random_state=42, reg_alpha=0, reg_lambda=1,
                                     scale_pos_weight=1, seed=None, silent=None,
                                     subsample=0.7, verbosity=1), param_grid=g_grid, scoring='roc_auc', n_jobs=-1, cv=3)
grid_7.fit(X_train_scaled, y_train)

In [ ]:
grid_7.best_params_, grid_7.best_score_

In [ ]:
h_grid = {'reg_alpha':[0.000001, 0.00001, 0.0001, 0.0002]} 

In [ ]:
grid_8 = GridSearchCV(estimator=XGBClassifier(base_score=0.5, booster='gbtree',
                                     colsample_bylevel=1, colsample_bynode=1,
                                     colsample_bytree=0.8, gamma=0.3,
                                     learning_rate=0.1, max_delta_step=0,
                                     max_depth=2, min_child_weight=3,
                                     missing=None, n_estimators=350, n_jobs=1,
                                     nthread=None, objective='binary:logistic',
                                     random_state=42, reg_alpha=0.0001, reg_lambda=1,
                                     scale_pos_weight=1, seed=None, silent=None,
                                     subsample=0.7, verbosity=1), param_grid=h_grid, scoring='roc_auc', n_jobs=-1, cv=3)
grid_8.fit(X_train_scaled, y_train)

In [ ]:
grid_8.best_params_, grid_8.best_score_

In [ ]:
xgb_tuned = XGBClassifier(base_score=0.5, booster='gbtree',
                                     colsample_bylevel=1, colsample_bynode=1,
                                     colsample_bytree=1, gamma=0,
                                     learning_rate=0.1, max_delta_step=0,
                                     max_depth=3, min_child_weight=1,
                                     missing=None, n_estimators=100, n_jobs=1,
                                     nthread=None, objective='binary:logistic',
                                     random_state=42, reg_alpha=0, reg_lambda=1,
                                     scale_pos_weight=1, seed=None, silent=None,
                                     subsample=1, verbosity=1)

In [ ]:
xgb_tuned.fit(X_train_scaled, y_train)

In [ ]:
predictions_tuned = xgb_tuned.predict(X_test_scaled)

In [ ]:
print(f'XGB model w/ scaling accuracy: {accuracy_score(y_test, predictions_tuned)}')
print(f'XGB model w/ scaling f1-score: {f1_score(y_test, predictions_tuned)}')

In [ ]:
final_tuned_pred = xgb_tuned.predict(df_test_scaled)

In [ ]:
submission_tuned = {'id':ids, 'predicted':final_tuned_pred}
xgb_tuned_submission = pd.DataFrame(submission_tuned)

In [ ]:
xgb_tuned_submission.to_csv('submission_11.csv', index=False)

# Treatment for test data

In [ ]:
df_test.head()

In [ ]:
df_test_copy = df_test
df_test_copy = df_test_copy.set_index('id')
df_test_copy.head(1)

In [ ]:
df_test_scaled = scaler.fit_transform(df_test_copy)
df_test_scaled = pd.DataFrame(df_test_scaled)
df_test_scaled.head(1)

In [ ]:
df_test_scaled.shape

In [ ]:
final_pred = xgb_model.predict(df_test)

In [ ]:
final_pred.shape

In [ ]:
ids = df_test['id']
submission = {'id':ids, 'predicted':final_pred}
df_submission = pd.DataFrame(submission)

In [ ]:
df_submission.shape

In [ ]:
df_submission.to_csv('submission_04.csv', index=False)

# Testando modelo com oversampling

In [ ]:
X_train_o, X_test_o, y_train_o, y_test_o = train_test_split(x_over, y_over, test_size=.33, random_state=5)

In [ ]:
X_train_o_scaled = scaler.fit_transform(X_train_o)
X_test_o_scaled = scaler.fit_transform(X_test_o)

In [ ]:
rfc.fit(X_train_o_scaled, y_train_o)
predictions_o = rfc.predict(X_test_o_scaled)
accuracy_score(y_test_o, predictions_o)

In [ ]:
metrics_df = pd.DataFrame({'f1_score':[f1_score(y_test, predictions),f1_score(y_test_o, predictions_o)],
                           'accuracy':[accuracy_score(y_test, predictions),accuracy_score(y_test_o, predictions_o)]
                           })
metrics_df

In [ ]:
final_pred_o = rfc.predict(df_test_scaled)
sns.countplot(final_pred_o)

In [ ]:
submission_over = {'id':ids, 'predicted':final_pred_o}
df_submission_o = pd.DataFrame(submission_over)

In [ ]:
df_submission_o.shape

In [ ]:
df_submission_o.to_csv('submission_02_1.csv', index=False)

In [ ]:
df_train_out_z.head()

In [ ]:
X_out = df_train_out_z.drop(['id', 'y'], axis = 1)
y_out = df_train_out_z['y']

In [ ]:
X_train_out, X_test_out, y_train_out, y_test_out = train_test_split(X_out, y_out, test_size=.33, random_state=5)

In [ ]:

scaler = StandardScaler()
X_train_scaled_out = scaler.fit_transform(X_train_out)
X_test_scaled_out = scaler.fit_transform(X_test_out)

In [ ]:
rfc_out = RandomForestClassifier(random_state=5)

In [ ]:
rfc_out.fit(X_train_scaled_out, y_train_out)
predictions_out = rfc_out.predict(X_test_out)
accuracy_score(y_test_out, predictions_out)